In [4]:
import pandas as pd
import numpy as np

USEFUL_COLS = [
    'Date', 'HomeTeam', 'AwayTeam', 'Referee',
    'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG',
    'HS', 'AS', 'HST', 'AST',
    'HC', 'AC', 'HF', 'AF',
    'HY', 'AY', 'HR', 'AR',
]

SEASONS = {
    '2526': '2025/26',
    '2425': '2024/25',
    '2324': '2023/24',
    '2223': '2022/23',
    '2122': '2021/22',
    '2021': '2020/21',
    '1920': '2019/20',
    '1819': '2018/19',
    '1718': '2017/18',
    '1617': '2016/17',
    '1516': '2015/16',
    '1415': '2014/15',
}

dfs = []
for code, name in SEASONS.items():
    url = f"https://www.football-data.co.uk/mmz4281/{code}/E0.csv"
    try:
        temp = pd.read_csv(url, usecols=lambda c: c in USEFUL_COLS)
        temp['Season'] = name
        dfs.append(temp)
        print(f"✅ {name} → {len(temp)} matches loaded")
    except Exception as e:
        print(f"❌ {name} → {e}")

df = pd.concat(dfs, ignore_index=True)
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)

# Drop future matches (current season)
df = df.dropna(subset=['FTR']).reset_index(drop=True)

print(f"\nTotal matches: {len(df)}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")

✅ 2025/26 → 349 matches loaded
✅ 2024/25 → 380 matches loaded
✅ 2023/24 → 380 matches loaded
✅ 2022/23 → 380 matches loaded
✅ 2021/22 → 380 matches loaded
✅ 2020/21 → 380 matches loaded
✅ 2019/20 → 380 matches loaded
✅ 2018/19 → 380 matches loaded
✅ 2017/18 → 380 matches loaded
✅ 2016/17 → 380 matches loaded
✅ 2015/16 → 380 matches loaded
✅ 2014/15 → 381 matches loaded

Total matches: 4529
Date range: 2014-08-16 00:00:00 to 2026-05-04 00:00:00


In [5]:
# Encode result into points for each team
def get_points(row, team):
    if row['FTR'] == 'H':
        return 3 if row['HomeTeam'] == team else 0
    elif row['FTR'] == 'A':
        return 3 if row['AwayTeam'] == team else 0
    else:
        return 1

# Build a flat list of matches per team
records = []
for _, row in df.iterrows():
    records.append({'Date': row['Date'], 'Team': row['HomeTeam'], 'Points': get_points(row, row['HomeTeam'])})
    records.append({'Date': row['Date'], 'Team': row['AwayTeam'],  'Points': get_points(row, row['AwayTeam'])})

team_df = pd.DataFrame(records).sort_values('Date').reset_index(drop=True)

# Rolling average of last 5 matches per team
team_df['form_last5'] = (
    team_df.groupby('Team')['Points']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

print(team_df.head(10))

        Date            Team  Points  form_last5
0 2014-08-16         Arsenal       3         NaN
1 2014-08-16  Crystal Palace       0         NaN
2 2014-08-16       Leicester       1         NaN
3 2014-08-16         Everton       1         NaN
4 2014-08-16      Man United       0         NaN
5 2014-08-16         Swansea       3         NaN
6 2014-08-16             QPR       0         NaN
7 2014-08-16            Hull       3         NaN
8 2014-08-16           Stoke       0         NaN
9 2014-08-16     Aston Villa       3         NaN


In [6]:
# Check that form_last5 fills in after first matchday
print(team_df[team_df['Team'] == 'Arsenal'].head(10))

          Date     Team  Points  form_last5
0   2014-08-16  Arsenal       3         NaN
22  2014-08-23  Arsenal       1    3.000000
57  2014-08-31  Arsenal       1    2.000000
60  2014-09-13  Arsenal       1    1.666667
89  2014-09-20  Arsenal       3    1.500000
101 2014-09-27  Arsenal       1    1.800000
139 2014-10-05  Arsenal       0    1.400000
142 2014-10-18  Arsenal       1    1.200000
161 2014-10-25  Arsenal       3    1.200000
192 2014-11-01  Arsenal       3    1.600000


In [7]:
# Merge form back to main df for home and away teams
home_form = team_df[['Date', 'Team', 'form_last5']].rename(
    columns={'Team': 'HomeTeam', 'form_last5': 'home_form_last5'}
)
away_form = team_df[['Date', 'Team', 'form_last5']].rename(
    columns={'Team': 'AwayTeam', 'form_last5': 'away_form_last5'}
)

df = df.merge(home_form, on=['Date', 'HomeTeam'], how='left')
df = df.merge(away_form, on=['Date', 'AwayTeam'], how='left')

# Quick check
print(df[['Date', 'HomeTeam', 'AwayTeam', 'FTR', 'home_form_last5', 'away_form_last5']].head(10))
print(f"\nNulls: {df[['home_form_last5', 'away_form_last5']].isnull().sum().to_dict()}")


        Date    HomeTeam        AwayTeam FTR  home_form_last5  away_form_last5
0 2014-08-16     Arsenal  Crystal Palace   H              NaN              NaN
1 2014-08-16   Leicester         Everton   D              NaN              NaN
2 2014-08-16  Man United         Swansea   A              NaN              NaN
3 2014-08-16         QPR            Hull   A              NaN              NaN
4 2014-08-16       Stoke     Aston Villa   A              NaN              NaN
5 2014-08-16   West Brom      Sunderland   D              NaN              NaN
6 2014-08-16    West Ham       Tottenham   A              NaN              NaN
7 2014-08-17   Newcastle        Man City   A              NaN              NaN
8 2014-08-17   Liverpool     Southampton   H              NaN              NaN
9 2014-08-18     Burnley         Chelsea   A              NaN              NaN

Nulls: {'home_form_last5': 18, 'away_form_last5': 17}


In [8]:
# --- REFEREE FEATURES ---
# Calculate referee stats BEFORE each match (no data leakage)

# Win rate for home team per referee
referee_stats = []

for idx, row in df.iterrows():
    ref = row['Referee']
    date = row['Date']

    # Only past matches with this referee
    past = df[(df['Referee'] == ref) & (df['Date'] < date)]

    if len(past) == 0:
        referee_stats.append({
            'ref_matches':        0,
            'ref_home_win_rate':  None,
            'ref_yellows_avg':    None,
            'ref_reds_avg':       None,
            'ref_goals_avg':      None,
        })
    else:
        home_wins = (past['FTR'] == 'H').sum()
        referee_stats.append({
            'ref_matches':        len(past),
            'ref_home_win_rate':  home_wins / len(past),
            'ref_yellows_avg':    (past['HY'] + past['AY']).mean(),
            'ref_reds_avg':       (past['HR'] + past['AR']).mean(),
            'ref_goals_avg':      (past['FTHG'] + past['FTAG']).mean(),
        })

ref_df = pd.DataFrame(referee_stats)
df = pd.concat([df, ref_df], axis=1)

print(df[['Referee', 'ref_matches', 'ref_home_win_rate', 'ref_yellows_avg', 'ref_goals_avg']].head(15))

          Referee  ref_matches  ref_home_win_rate  ref_yellows_avg  \
0          J Moss            0                NaN              NaN   
1         M Jones            0                NaN              NaN   
2          M Dean            0                NaN              NaN   
3        C Pawson            0                NaN              NaN   
4        A Taylor            0                NaN              NaN   
5     N Swarbrick            0                NaN              NaN   
6           C Foy            0                NaN              NaN   
7      M Atkinson            0                NaN              NaN   
8   M Clattenburg            0                NaN              NaN   
9        M Oliver            0                NaN              NaN   
10  M Clattenburg            1                1.0              3.0   
11         M Dean            1                0.0              6.0   
12       C Pawson            1                0.0              3.0   
13        L Mason   

In [9]:
# --- GOALS SCORED / CONCEDED AVERAGES (last 5 matches) ---

goal_records = []
for _, row in df.iterrows():
    goal_records.append({
        'Date': row['Date'], 'Team': row['HomeTeam'],
        'scored': row['FTHG'], 'conceded': row['FTAG']
    })
    goal_records.append({
        'Date': row['Date'], 'Team': row['AwayTeam'],
        'scored': row['FTAG'], 'conceded': row['FTHG']
    })

goals_df = pd.DataFrame(goal_records).sort_values('Date').reset_index(drop=True)

goals_df['avg_scored_last5'] = (
    goals_df.groupby('Team')['scored']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)
goals_df['avg_conceded_last5'] = (
    goals_df.groupby('Team')['conceded']
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

# Merge back for home and away
home_goals = goals_df[['Date', 'Team', 'avg_scored_last5', 'avg_conceded_last5']].rename(
    columns={'Team': 'HomeTeam',
             'avg_scored_last5': 'home_avg_scored',
             'avg_conceded_last5': 'home_avg_conceded'}
)
away_goals = goals_df[['Date', 'Team', 'avg_scored_last5', 'avg_conceded_last5']].rename(
    columns={'Team': 'AwayTeam',
             'avg_scored_last5': 'away_avg_scored',
             'avg_conceded_last5': 'away_avg_conceded'}
)

df = df.merge(home_goals, on=['Date', 'HomeTeam'], how='left')
df = df.merge(away_goals, on=['Date', 'AwayTeam'], how='left')

print(df[['HomeTeam', 'AwayTeam', 'FTR',
          'home_avg_scored', 'home_avg_conceded',
          'away_avg_scored', 'away_avg_conceded']].head(10))

     HomeTeam        AwayTeam FTR  home_avg_scored  home_avg_conceded  \
0     Arsenal  Crystal Palace   H              NaN                NaN   
1   Leicester         Everton   D              NaN                NaN   
2  Man United         Swansea   A              NaN                NaN   
3         QPR            Hull   A              NaN                NaN   
4       Stoke     Aston Villa   A              NaN                NaN   
5   West Brom      Sunderland   D              NaN                NaN   
6    West Ham       Tottenham   A              NaN                NaN   
7   Newcastle        Man City   A              NaN                NaN   
8   Liverpool     Southampton   H              NaN                NaN   
9     Burnley         Chelsea   A              NaN                NaN   

   away_avg_scored  away_avg_conceded  
0              NaN                NaN  
1              NaN                NaN  
2              NaN                NaN  
3              NaN                Na

In [10]:
# Check a later match to verify values are filling in
print(df[['HomeTeam', 'AwayTeam', 'FTR',
          'home_avg_scored', 'home_avg_conceded',
          'away_avg_scored', 'away_avg_conceded']].iloc[50:55])

          HomeTeam     AwayTeam FTR  home_avg_scored  home_avg_conceded  \
50      Sunderland      Swansea   D              1.0                1.2   
51         Chelsea  Aston Villa   H              3.2                1.4   
52  Crystal Palace    Leicester   H              1.6                2.0   
53            Hull     Man City   A              1.4                1.4   
54       Liverpool      Everton   D              1.4                1.6   

    away_avg_scored  away_avg_conceded  
50              1.6                1.2  
51              0.8                0.8  
52              1.8                1.6  
53              1.6                1.0  
54              2.2                2.6  


In [13]:
# --- CALCULATE ELO FROM SCRATCH ---
K = 32        # sensitivity, standard value
BASE_ELO = 1500

elo_dict = {}
elo_records = []

for _, row in df.sort_values('Date').iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']

    # Initialize if first time seeing team
    if home not in elo_dict:
        elo_dict[home] = BASE_ELO
    if away not in elo_dict:
        elo_dict[away] = BASE_ELO

    elo_home = elo_dict[home]
    elo_away = elo_dict[away]

    # Expected win probability
    expected_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))
    expected_away = 1 - expected_home

    # Actual result
    if row['FTR'] == 'H':
        actual_home, actual_away = 1, 0
    elif row['FTR'] == 'A':
        actual_home, actual_away = 0, 1
    else:
        actual_home, actual_away = 0.5, 0.5

    # Save ELO BEFORE updating (no leakage)
    elo_records.append({
        'Date':     row['Date'],
        'HomeTeam': home,
        'AwayTeam': away,
        'elo_home': elo_home,
        'elo_away': elo_away,
        'elo_diff': elo_home - elo_away,
    })

    # Update ELO after match
    elo_dict[home] = elo_home + K * (actual_home - expected_home)
    elo_dict[away] = elo_away + K * (actual_away - expected_away)

elo_df = pd.DataFrame(elo_records)
df = df.merge(elo_df, on=['Date', 'HomeTeam', 'AwayTeam'], how='left')

print(df[['HomeTeam', 'AwayTeam', 'elo_home', 'elo_away', 'elo_diff']].iloc[50:55])

          HomeTeam     AwayTeam     elo_home     elo_away   elo_diff
50      Sunderland      Swansea  1482.720925  1512.556114 -29.835189
51         Chelsea  Aston Villa  1559.631577  1528.453773  31.177804
52  Crystal Palace    Leicester  1486.391363  1518.813992 -32.422629
53            Hull     Man City  1496.524849  1516.064087 -19.539237
54       Liverpool      Everton  1485.178098  1484.547750   0.630348


In [17]:
# --- HEAD TO HEAD FEATURES ---

h2h_records = []

for idx, row in df.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']
    date = row['Date']

    # Get all past matches between these two teams
    past = df[
        (df['Date'] < date) &
        (
            ((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) |
            ((df['HomeTeam'] == away) & (df['AwayTeam'] == home))
        )
    ].tail(5)  # last 5 encounters only

    if len(past) == 0:
        h2h_records.append({
            'h2h_matches':   0,
            'h2h_home_wins': None,
            'h2h_draws':     None,
            'h2h_goals_avg': None,
        })
    else:
        # Count wins from home team perspective
        home_wins = 0
        draws = 0
        for _, p in past.iterrows():
            if p['HomeTeam'] == home:
                if p['FTR'] == 'H': home_wins += 1
                elif p['FTR'] == 'D': draws += 1
            else:
                if p['FTR'] == 'A': home_wins += 1
                elif p['FTR'] == 'D': draws += 1

        h2h_records.append({
            'h2h_matches':   len(past),
            'h2h_home_wins': home_wins / len(past),
            'h2h_draws':     draws / len(past),
            'h2h_goals_avg': (past['FTHG'] + past['FTAG']).mean(),
        })

h2h_df = pd.DataFrame(h2h_records)
df = pd.concat([df, h2h_df], axis=1)

# Quick check
print(df[['HomeTeam', 'AwayTeam', 'h2h_matches', 'h2h_home_wins', 'h2h_draws', 'h2h_goals_avg']].iloc[50:55])

          HomeTeam     AwayTeam  h2h_matches  h2h_home_wins  h2h_draws  \
50      Sunderland      Swansea            0            NaN        NaN   
51         Chelsea  Aston Villa            0            NaN        NaN   
52  Crystal Palace    Leicester            0            NaN        NaN   
53            Hull     Man City            0            NaN        NaN   
54       Liverpool      Everton            0            NaN        NaN   

    h2h_goals_avg  
50            NaN  
51            NaN  
52            NaN  
53            NaN  
54            NaN  


In [18]:
# Check a recent match with enough history
print(df[['HomeTeam', 'AwayTeam', 'h2h_matches', 'h2h_home_wins', 'h2h_draws', 'h2h_goals_avg']].iloc[200:205])

           HomeTeam     AwayTeam  h2h_matches  h2h_home_wins  h2h_draws  \
200         Burnley          QPR            1            0.0        0.0   
201  Crystal Palace    Tottenham            1            0.0        1.0   
202         Everton     Man City            1            0.0        0.0   
203       Leicester  Aston Villa            1            0.0        0.0   
204      Sunderland    Liverpool            1            0.0        1.0   

     h2h_goals_avg  
200            2.0  
201            0.0  
202            1.0  
203            3.0  
204            0.0  


In [19]:
# --- FINAL DATASET ---
FEATURE_COLS = [
    'home_form_last5',
    'away_form_last5',
    'home_avg_scored',
    'home_avg_conceded',
    'away_avg_scored',
    'away_avg_conceded',
    'ref_matches',
    'ref_home_win_rate',
    'ref_yellows_avg',
    'ref_reds_avg',
    'ref_goals_avg',
    'elo_home',
    'elo_away',
    'elo_diff',
    'h2h_matches',
    'h2h_home_wins',
    'h2h_draws',
    'h2h_goals_avg',
]

EXTRA_COLS = ['Referee', 'FTHG', 'FTAG', 'HY', 'AY', 'HR', 'AR']

df['target'] = df['FTR'].map({'H': 0, 'D': 1, 'A': 2})

df_model = df[FEATURE_COLS + EXTRA_COLS + ['target', 'Date', 'HomeTeam', 'AwayTeam', 'Season']].dropna(subset=FEATURE_COLS).reset_index(drop=True)

print(f"Total rows for model: {len(df_model)}")
print(f"Nulls remaining: {df_model[FEATURE_COLS].isnull().sum().sum()}")
print(f"\nTarget distribution:")
print(df_model['target'].value_counts(normalize=True).round(3))

df_model.to_csv('data/features.csv', index=False)
print("\n✅ Saved to data/features.csv")

Total rows for model: 4004
Nulls remaining: 0

Target distribution:
target
0    0.446
2    0.320
1    0.234
Name: proportion, dtype: float64

✅ Saved to data/features.csv
